# Dataset Selection & Feature-Type Taxonomy

**Week 2 task (22–29 Jun 2026)** — Phase A, Milestone M2  
Select 3–4 tabular datasets (OpenML/UCI) varying in size and dimensionality, then classify each by feature type and compute mutual information (MI) with class labels. This taxonomy drives stratified reporting in Phase C.

## Selection criteria
1. Ground-truth cluster labels available  
2. Varying row count and feature count  
3. Spans the categorical-fraction spectrum (0 % → 100 %) to enable stratified reporting  
4. Well-known in the tabular ML literature (reproducibility / comparability)  

## Chosen datasets

| slug | Name | OpenML ID | Rows | Features | Classes | Cat% | Stratum |
|------|------|-----------|------|----------|---------|------|---------|
| iris | Iris | 61 | 150 | 4 | 3 | 0 % | all-numerical |
| credit-g | German Credit | 31 | 1 000 | 20 | 2 | 65 % | high-categorical |
| car | Car Evaluation | 40975 | 1 728 | 6 | 4 | 100 % | all-categorical |
| adult | Adult/Census Income | 1590 | 48 842 | 14 | 2 | 57 % | high-categorical |

**Stratification axis for Phase C**: all-numerical (Iris) vs. high-categorical (German Credit, Car, Adult).  
The key hypothesis: generic corruption augmentation (SCARF, Gaussian noise, feature swap) should degrade NMI/ARI more on the high-categorical stratum than on the all-numerical stratum.  
The Type-Aware pipeline should close that gap.

> **Note on Adult**: 48 k rows may be slow without GPU. Use Google Colab (Phase C) or subsample to 10 k rows with a fixed seed if training time is a blocker in Phase B.

In [ ]:
import ssl, certifi
import os
# Fix macOS SSL cert verification for sklearn's OpenML fetcher
os.environ['SSL_CERT_FILE'] = certifi.where()

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
MANIFEST = [
    ('iris',     61,    'Iris'),
    ('credit-g', 31,    'German Credit'),
    ('car',      40975, 'Car Evaluation'),
    ('adult',    1590,  'Adult/Census Income'),
]

datasets = {}
for slug, oid, name in MANIFEST:
    print(f'Fetching {name}...')
    ds = fetch_openml(data_id=oid, as_frame=True, parser='auto')
    datasets[slug] = {'name': name, 'openml_id': oid, 'X': ds.data.copy(), 'y': ds.target.copy()}

print('Done.')

## Feature-type taxonomy

In [ ]:
def classify_features(X):
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    return num_cols, cat_cols


def compute_mi(X, y, cat_cols):
    X_enc = X.copy()
    for c in cat_cols:
        X_enc[c] = LabelEncoder().fit_transform(X_enc[c].astype(str))
    X_enc = X_enc.fillna(X_enc.median(numeric_only=True))
    y_enc = LabelEncoder().fit_transform(y.astype(str))
    discrete_mask = [c in cat_cols for c in X_enc.columns]
    mi = mutual_info_classif(X_enc, y_enc, discrete_features=discrete_mask, random_state=0)
    return pd.Series(mi, index=X_enc.columns).sort_values(ascending=False)


records = []
mi_results = {}

for slug, oid, name in MANIFEST:
    X, y = datasets[slug]['X'], datasets[slug]['y']
    num_cols, cat_cols = classify_features(X)
    n_rows, n_cols = X.shape
    n_num, n_cat = len(num_cols), len(cat_cols)
    cat_frac = n_cat / n_cols

    mi = compute_mi(X, y, cat_cols)
    mi_results[slug] = mi

    stratum = (
        'all-numerical'   if cat_frac == 0.0 else
        'all-categorical' if cat_frac == 1.0 else
        'high-categorical'
    )

    records.append({
        'slug':           slug,
        'name':           name,
        'openml_id':      oid,
        'n_rows':         n_rows,
        'n_features':     n_cols,
        'n_classes':      y.nunique(),
        'n_numerical':    n_num,
        'n_categorical':  n_cat,
        'cat_fraction':   round(cat_frac, 2),
        'stratum':        stratum,
        'top_mi_feature': mi.index[0],
        'top_mi_value':   round(float(mi.iloc[0]), 4),
    })

taxonomy = pd.DataFrame(records).set_index('slug')
taxonomy

## MI per feature — all datasets

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

palette = {'num': '#4878CF', 'cat': '#D65F5F'}

for ax, (slug, oid, name) in zip(axes, MANIFEST):
    mi = mi_results[slug]
    _, cat_cols = classify_features(datasets[slug]['X'])
    colors = [palette['cat'] if f in cat_cols else palette['num'] for f in mi.index]
    ax.barh(mi.index[::-1], mi.values[::-1], color=colors[::-1])
    ax.set_title(f'{name}  (cat={taxonomy.loc[slug,"cat_fraction"]:.0%})', fontsize=11)
    ax.set_xlabel('MI with class label')
    ax.axvline(0, color='black', linewidth=0.8)

# legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=palette['num'], label='numerical'),
              Patch(facecolor=palette['cat'], label='categorical')]
fig.legend(handles=legend_els, loc='lower center', ncol=2, fontsize=10)
fig.suptitle('Mutual Information with class label — per dataset', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('dataset_mi_taxonomy.png', dpi=150, bbox_inches='tight')
plt.show()

## Key observations for Phase C stratification

**Iris (0 % categorical, all-numerical)**  
`petallength` (MI = 0.99) and `petalwidth` (MI = 0.97) almost perfectly separate the three classes. Even moderate numerical corruption risks destroying this signal. Baseline for testing whether augmentation helps or hurts when features are purely numerical.

**German Credit (65 % categorical)**  
All MI values are low (max 0.07). The top signal is categorical (`checking_status`). This is a hard clustering task — corruption of categorical features will likely push already-weak signal below noise floor. Key test case for Type-Aware vs. generic augmentation.

**Car Evaluation (100 % categorical)**  
`safety` and `persons` carry most signal (MI 0.18, 0.15). The 6-feature, all-categorical structure makes this the clearest test of whether corrupting categorical features destroys cluster structure. Generic augmentation (Gaussian noise, feature swap applied to ordinal-encoded integers) is by definition inapplicable here — Type-Aware must handle it.

**Adult/Census Income (57 % categorical, large)**  
The top two features are categorical (`relationship` MI 0.11, `marital-status` MI 0.11). Numerical features (`capital-gain`, `age`) follow. This is the thesis's clearest example of the semantic-validity concern: corrupting `marital-status` or `relationship` creates a semantically different entity, not a different view of the same person. Scale (48 k rows) also tests whether pipeline is efficient enough for practical use.

## Export taxonomy CSV

In [ ]:
taxonomy.to_csv('dataset_taxonomy.csv')
print('Saved dataset_taxonomy.csv')
taxonomy[['name','n_rows','n_features','n_classes','n_numerical','n_categorical','cat_fraction','stratum','top_mi_feature','top_mi_value']]